# 2주차 과제 - Google Colab 버전

이 노트북은 로컬 환경에서 문제가 있을 때 Google Colab에서 실행할 수 있는 버전입니다.


In [ ]:
# 패키지 설치
!pip install opencv-python numpy

import cv2
import numpy as np
import os
from google.colab import files

print(f"OpenCV 버전: {cv2.__version__}")
print(f"NumPy 버전: {np.__version__}")


In [1]:
# 이미지 업로드 (files import 필요)
from google.colab import files

print("OIP.jpg 파일을 업로드해주세요:")
uploaded = files.upload()

# 업로드된 파일 확인
for filename in uploaded.keys():
    print(f"업로드된 파일: {filename}")
    print(f"파일 크기: {len(uploaded[filename])} bytes")


OIP.jpg 파일을 업로드해주세요:


Saving OIP.jpg to OIP.jpg
업로드된 파일: OIP.jpg
파일 크기: 20052 bytes


In [2]:
# 2D → 3D 변환 함수들
def generate_depth_map(image):
    """이미지에서 Depth Map 생성"""
    if image is None:
        raise ValueError("이미지가 None입니다.")

    if len(image.shape) != 3:
        raise ValueError(f"예상: 3차원 이미지, 실제: {len(image.shape)}차원")

    # 그레이스케일 변환
    grayscale = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # JET 컬러맵 적용 (Depth Map 효과)
    depth_map = cv2.applyColorMap(grayscale, cv2.COLORMAP_JET)

    return depth_map

def create_3d_point_cloud(image, depth_map):
    """Depth Map에서 3D 포인트 클라우드 생성"""
    if image is None or depth_map is None:
        raise ValueError("이미지나 Depth Map이 None입니다.")

    h, w = image.shape[:2]

    # 그레이스케일 변환
    grayscale = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 3D 좌표 그리드 생성
    X, Y = np.meshgrid(np.arange(w), np.arange(h))

    # 모든 좌표를 float32로 변환
    X = X.astype(np.float32)
    Y = Y.astype(np.float32)

    # Depth 값을 Z 축으로 사용 (정규화)
    Z = grayscale.astype(np.float32) / 255.0 * 100  # 0-100 범위로 스케일링

    # 3D 좌표 생성
    points_3d = np.dstack((X, Y, Z))

    return points_3d

print("✅ 2D → 3D 변환 함수 정의 완료")


✅ 2D → 3D 변환 함수 정의 완료


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# 이미지 처리 실행
import cv2
import numpy as np

try:
    # 이미지 로드
    image = cv2.imread('OIP.jpg')
    if image is None:
        print("❌ OIP.jpg 로드 실패")
    else:
        print(f"✅ 이미지 로드 성공: {image.shape}")

        # Depth Map 생성
        depth_map = generate_depth_map(image)
        print(f"✅ Depth Map 생성: {depth_map.shape}")

        # 3D 포인트 클라우드 생성
        points_3d = create_3d_point_cloud(image, depth_map)
        print(f"✅ 3D 포인트 클라우드 생성: {points_3d.shape}")

        # 결과 저장
        cv2.imwrite('OIP_depth_map.jpg', depth_map)

        # 3D 포인트 클라우드 저장
        points_flat = points_3d.reshape(-1, 3)
        np.savetxt('OIP_3d_points.txt', points_flat, fmt='%.6f', header='X Y Z', comments='')

        print("\n🎉 처리 완료!")
        print("생성된 파일:")
        print("- OIP_depth_map.jpg (Depth Map)")
        print("- OIP_3d_points.txt (3D 포인트 클라우드)")

except Exception as e:
    print(f"❌ 오류 발생: {e}")
    import traceback
    traceback.print_exc()

✅ 이미지 로드 성공: (266, 474, 3)
✅ Depth Map 생성: (266, 474, 3)
✅ 3D 포인트 클라우드 생성: (266, 474, 3)

🎉 처리 완료!
생성된 파일:
- OIP_depth_map.jpg (Depth Map)
- OIP_3d_points.txt (3D 포인트 클라우드)


In [8]:
# 결과 파일 다운로드
from google.colab import files

print("결과 파일 다운로드:")
files.download('OIP_depth_map.jpg')
files.download('OIP_3d_points.txt')
print("✅ 다운로드 완료!")

결과 파일 다운로드:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 다운로드 완료!
